# From Transcription to Subject Metadata
### A hands-on introduction to automated subject indexing with Annif, using early modern normative texts from Bern (1528–1798)

**Notebook by:** C. Annemieke Romein (University of Twente, Enschede, the Netherlands) and Jona M. Hassenbach (University of Bern, Bern, Switzerland)  
**Workshop:** HISTMET & Annif, Part Two: Annif · [Part One: HISTMET](https://colab.research.google.com/github/CARomein/WorkshopBern2026_HISTMET_ANNIF/blob/master/01_HISTMET_text_length.ipynb)  
**Version:** 1.0 (September 2026) · **DOI:** *to be added upon deposit in Zenodo* · **Licence:** *to be added*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CARomein/WorkshopBern2026_HISTMET_ANNIF/blob/master/02_Annif_subject_indexing.ipynb)

## About this notebook

Archives increasingly hold large volumes of transcribed (ATR/HTR) historical text. Such text is *searchable*, but not necessarily *findable*: early modern spelling, terminology and sheer volume make it very hard to locate, say, every ordinance on market regulation. Subject labels drawn from a controlled vocabulary can close that gap. [Annif](https://annif.org), an open-source toolkit developed by the National Library of Finland, assigns such labels automatically; it is lightweight, runs without a GPU, and its output is stable and auditable metadata.

Part One showed that unsupervised clustering recovers subjects only partially. In this notebook you will train an Annif model on early modern normative texts (*Policeyordnungen*, the regulatory ordinances by which early modern authorities governed public order, economic life and social conduct), evaluate it, discover why Annif's default settings are misleading, and compare two transcription versions of the same corpus. The notebook accompanies the article by Hassenbach and Romein, *Bringing Subject Indexing to the Archive: Does Improved Transcription Quality Necessitate Metadata Regeneration? Evidence from Annif and Early Modern Normative Texts* (forthcoming).

### The data

The corpus consists of 4,930 normative texts from the City-Republic of Bern, manually labelled by Claudia Schott-Volm (2006) following the classification of the *Repertorium der Policeyordnungen* of the Max Planck Institute. That classification is encoded as a hierarchical SKOS vocabulary; we work with Level 2 (5 categories), Level 3 (25 categories) and Level 4 (c. 130 categories). Each text is available in two uncorrected transcriptions: a **2020** version (CITlab HTR+, character error rate c. 7.5%) and a **2024** version (Transkribus Text Titan I, c. 2.95%). The article used 10-fold cross-validation; to keep runtimes short, this notebook uses only the first fold (*fold 0*), i.e. c. 4,438 documents for training and 492 for testing.

### How to use this notebook

Run the cells from top to bottom, either with the ▶ button or with `Shift + Enter`. Everything runs on Google's servers; a standard CPU runtime suffices (no GPU is needed). Lines beginning with `!` are command-line instructions rather than Python, and expressions in curly brackets such as `{year}` are filled in with the Python variables you set in Section 4. Note that a notebook opened from GitHub is not saved anywhere: choose *File → Save a copy in Drive* at the start if you wish to keep your edits and results. Colab deletes all files when the session ends, so download anything you want to keep.

| Section | What happens |
|---|---|
| 1. Set-up | Install Annif and its optional backends |
| 2. Data import | Download the corpus, vocabularies and configuration |
| 3. Loading Annif | Check that the installation works |
| 4. Model training and evaluation | Train a model, evaluate it, tune *Limit* and *Threshold* |
| 5. Comparing results | Plot your own results |
| 6. Our results | Inspect the full set of results from the article |
| 7. Development over time | *(under construction)* |

# **1. SET-UP**

The first cell installs Annif itself. Installation takes a few minutes; red warnings about dependency conflicts with packages pre-installed in Colab can safely be ignored. Should Colab ask you to restart the session, do so and continue with the next cell (there is no need to rerun this one).

In [ ]:
!pip install annif

Annif offers several classification algorithms (*backends*), some of which require additional packages. The next cell installs `omikuji` (needed for the Parabel and Bonsai backends) and `nn` (needed for the neural ensemble; this downloads TensorFlow, c. 260 MB, and therefore takes a little while).

In [ ]:
!pip install annif[omikuji]
!pip install annif[nn]

# **2. DATA IMPORT**

This cell copies the workshop repository from GitHub into Colab's working folder (`/content`) and removes Colab's own sample data. Among the files you will receive are `projects.cfg`, which defines every Annif *project*, i.e. one combination of transcription, vocabulary level and backend (for example `trans_2020_level2_omikuji_parabel`); `data/vocabs`, holding the three vocabulary levels, already loaded into Annif; the folders `trans_<year>_level<n>/fold_0/train` and `/test`, which contain, for each document, a `.txt` file with the transcription and a `.tsv` file with its manually assigned labels; the folders `results_default` and `results_optimal`, holding our own results (Section 6); and `visualize_simple.py`, a small plotting script.

In [ ]:
!rm -rf /content/sample_data
!git clone https://github.com/CARomein/WorkshopBern2026_HISTMET_ANNIF.git /content/_tmp_repo
!mv /content/_tmp_repo/* /content/_tmp_repo/.[!.]* /content/ 2>/dev/null
!rm -rf /content/_tmp_repo

This cell creates two empty folders in which the results of your own evaluations will be stored.

In [ ]:
import os
for d in ["results/trans_2020", "results/trans_2024"]:
    os.makedirs(d, exist_ok=True)

# **3. LOADING ANNIF**

Importing Annif in Python merely confirms that the installation succeeded. The actual work is done through Annif's command-line commands `annif train`, `annif eval` and `annif optimize`.

In [ ]:
import annif

# **4. MODEL TRAINING AND EVALUATION**

Here you choose which experiment to run by setting three variables: the transcription version (`year`), the classification algorithm (`backend`) and the vocabulary level (`level`). Together they determine which Annif project and which data folder the subsequent cells use. To try another combination, change the values and rerun the cells from here onwards.

| Variable | Options | Meaning |
|---|---|---|
| `year` | `2020`, `2024` | older (noisier) or newer (better) transcription |
| `level` | `2`, `3`, `4` | 5, 25 or c. 130 categories |
| `backend` | `tfidf`, `svc`, `omikuji_parabel`, `omikuji_bonsai` | statistical backends: they learn which words co-occur with which labels |
| | `mllm`, `yake` | lexical backends: they look for the vocabulary's (modern German) label terms in the text |
| | `nn_ensemble` | a neural network combining the predictions of the other backends |

Expect the lexical backends to score near zero, since early modern spelling rarely matches the modern label terms. The `nn_ensemble` can only be trained once all of the backends it combines have been trained for the same year and level.

In [ ]:
year = 2020                 #alternatives: 2024
backend = "omikuji_parabel" #alternatives: tfidf omikuji_bonsai svc mllm yake
level = 2                   #alternatives: 3 4


### 4.1 Training

`annif train` feeds the training documents and their manual labels to the chosen backend, which learns associations between words and labels. For most backends this takes between a few seconds and a minute; the log output merely documents progress.

In [ ]:
!annif train trans_{year}_level{level}_{backend} trans_{year}_level{level}/fold_0/train

### 4.2 Evaluation with default settings

`annif eval` applies the trained model to the 492 test documents it has not seen before and compares its suggestions with the manual labels. The key figures are *precision* (how many suggested labels are correct), *recall* (how many correct labels were found) and *F1* (the harmonic mean of the two); we use the *microavg* values, as in the article. Note that recall is 1.0 and precision very low: with its default settings, Annif returns every label in the vocabulary for every document. The result is stored in `results/trans_<year>/`.

In [ ]:
!annif eval trans_{year}_level{level}_{backend} trans_{year}_level{level}/fold_0/test --results-file results/trans_{year}/trans_{year}_level{level}_{backend}_fold0.json

### 4.3 Evaluation with Limit and Threshold set

Two parameters control Annif's output. **Limit** caps the number of labels returned per document; **Threshold** sets the minimum confidence score (between 0 and 1) a label needs in order to be returned. Most documents in this corpus carry only one or two labels, so sensible values make a considerable difference. Set values in the first cell, run the evaluation, and compare the F1 score with that of 4.2. Be aware that this evaluation overwrites the result file of 4.2.

In [ ]:
threshold = 0.2
limit = 3

In [ ]:
!annif eval --threshold={threshold} --limit={limit} trans_{year}_level{level}_{backend} trans_{year}_level{level}/fold_0/test --results-file results/trans_{year}/trans_{year}_level{level}_{backend}_fold0.json

### 4.4 Letting Annif optimise

Rather than guessing, `annif optimize` evaluates every combination of Limit (1 to 15) and Threshold (0 to 0.95) and reports the best one for precision, recall and F1. The table columns are Limit, Threshold, precision, recall and F1. Two caveats apply. First, for demonstration purposes we optimise on the test set itself; in a proper study one would use a separate validation set, as optimising on the test data yields overly optimistic scores. Second, `optimize` reports document-averaged scores, which differ slightly from the micro-averaged scores used elsewhere.

In [ ]:
!annif optimize trans_{year}_level{level}_{backend} trans_{year}_level{level}/fold_0/test

# **5. COMPARING RESULTS**

The function `vis_results` collects all result files in the folder `results` and plots the F1 score per backend and vocabulary level, for the 2020 and 2024 transcriptions side by side.

**Careful!** The plot simply uses whichever result file was saved last for each combination, irrespective of the settings used. You might thus unwittingly be comparing a default-setting Parabel with an optimised TF-IDF; keep your settings consistent across the runs you wish to compare.

*Micro*-averaging pools all correct and incorrect suggestions across labels, so that frequent categories weigh most heavily; *macro*-averaging computes F1 per label and then averages, so that rare categories count as much as frequent ones. Macro scores are therefore usually lower on unevenly distributed data such as ours.

In [ ]:
from visualize_simple import vis_results
micro_avg = "micro"
macro_avg = "macro"

In [ ]:
vis_results("results", micro_avg)

# **6. OUR RESULTS**

These plots show our own results for all seven backends, all three levels and both transcriptions (fold 0): first under default settings, then under the optimised configuration reported in the article (Limit = 2, Threshold = 0.1), and finally the optimised results macro-averaged. When comparing them, consider which matters more, the transcription version or the configuration, and why the lexical backends fail.

In [ ]:
vis_results("results_default", micro_avg)

In [ ]:
vis_results("results_optimal", micro_avg)

In [ ]:
vis_results("results_optimal", macro_avg)

# **7. DEVELOPMENT OVER TIME**

*This section is under construction.* It will explore how classification performance develops across the period covered by the corpus (1528–1798).

# **FURTHER READING AND CITATION**

Documentation of Annif, including all backends and commands: [annif.org](https://annif.org) and the [Annif wiki on GitHub](https://github.com/NatLibFi/Annif/wiki).

On the vocabulary: C. Annemieke Romein, Andreas Wagner and Karl Härter, 'The 2020-Systematic, Multilingual Categorisation of Policeymaterien for Early Modern Police Ordinances', *Journal for Digital Legal History* 4, no. 1 (2025), [doi:10.21825/dlh.93163](https://doi.org/10.21825/dlh.93163).

On the Bernese corpus and earlier Annif experiments with it: Christel Annemieke Romein, Sara F. Veldhoen and Jaap C. Romein, 'Applying (semi-)automatic metadata to early modern normative texts. Annif and Policeygesetzgebung from the City-State of Bern (1528–1798)', *Digital Scholarship in the Humanities* 41, Suppl. 1 (2026), i230–i237, [doi:10.1093/llc/fqaf072](https://doi.org/10.1093/llc/fqaf072).

The experiment is described in full in Jona M. Hassenbach and C. Annemieke Romein, *Bringing Subject Indexing to the Archive: Does Improved Transcription Quality Necessitate Metadata Regeneration? Evidence from Annif and Early Modern Normative Texts* (forthcoming).

To cite this notebook: C. Annemieke Romein and Jona M. Hassenbach, *From Transcription to Subject Metadata: A Hands-On Introduction to Automated Subject Indexing with Annif* (version 1.0, 2026), Zenodo, DOI: *to be added*.